In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================================================
# SDXL BASE + REFINER RELATIONAL IMAGE GENERATION
# (with Ground-Truth Annotation + Proper Latent Passing)
# =========================================================

import torch, os, random, gc, shutil
from tqdm import tqdm
from PIL import Image
import pandas as pd
from diffusers import DiffusionPipeline, AutoencoderKL

# ------------------------
# CONFIGURATION  (DEFINE FIRST!)
# ------------------------
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

total_images = 1
seed_start = 42
n_steps = 50
guidance_scale = 7.5
high_noise_frac = 0.4

# ------------------------
# DEVICE
# ------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print("Using device:", device)

# ------------------------
# LOAD VAE (VERY IMPORTANT FOR SDXL)
# ------------------------
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch_dtype
)

# ------------------------
# LOAD BASE PIPELINE
# ------------------------
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch_dtype,
    variant="fp16",
    use_safetensors=True
)

pipe.to(device)
pipe.enable_attention_slicing()
pipe.enable_model_cpu_offload()

# ------------------------
# LOAD REFINER PIPELINE
# ------------------------
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    vae=vae,
    torch_dtype=torch_dtype,
    variant="fp16",
    use_safetensors=True
)

refiner.to(device)
refiner.enable_model_cpu_offload()

# ------------------------
# OBJECTS & RELATIONS
# ------------------------
OBJECTS = [
    "cat","dog","horse","car","bicycle","bus",
    "apple","banana","chair","table",
    "bird","boat","airplane","cow","sheep",
    "tv","laptop","phone","cup"
]

RELATIONS = [
    "slightly larger than",
    "larger than",
    "much larger than",
    "slightly smaller than",
    "smaller than",
    "taller than",
    "shorter than"
]

# =========================================================
# PROMPT GENERATION
# =========================================================
def generate_prompts(n):
    prompts = []
    metadata = []

    for _ in range(n):
        obj_a, obj_b = random.sample(OBJECTS, 2)
        relation = random.choice(RELATIONS)

        prompt = (
            f" a {obj_a} is {relation} a {obj_b}"
        )

        prompts.append(prompt)
        metadata.append({
            "object_A": obj_a,
            "object_B": obj_b,
            "relation": relation
        })

    return prompts, metadata


prompts, meta_info = generate_prompts(total_images)

records = []

# =========================================================
# IMAGE GENERATION (CORRECT SDXL TWO-STAGE)
# =========================================================
def generate_image(prompt, seed, idx):

    generator = torch.Generator(device=device).manual_seed(seed)

    # ---- BASE MODEL (produces latents + embeddings) ----
    base_output = pipe(
        prompt=prompt,
        num_inference_steps=n_steps,
        denoising_end=high_noise_frac,
        guidance_scale=guidance_scale,
        generator=generator,
        output_type="latent"
    )

    latents = base_output.images

    # ---- REFINER MODEL (uses SAME embeddings!) ----
    image = refiner(
        prompt=prompt,
        num_inference_steps=n_steps,
        denoising_start=high_noise_frac,
        guidance_scale=guidance_scale,
        generator=generator,
        image=latents
    ).images[0]

    # Save
    filename = f"generated_{idx:05d}.png"
    save_path = os.path.join(output_dir, filename)
    image.save(save_path)

    # Save annotation
    records.append({
        "image": filename,
        "prompt": prompt,
        "seed": seed,
        "object_A": meta_info[idx]["object_A"],
        "object_B": meta_info[idx]["object_B"],
        "relation": meta_info[idx]["relation"]
    })

    # Memory cleanup
    del image, latents, base_output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================================================
# GENERATE DATASET
# =========================================================
# for i in tqdm(range(total_images), desc="Generating images"):
#     generate_image(prompts[i], seed_start + i, i)

# =========================================================
# SAVE CSV (GROUND TRUTH!)
# =========================================================
df = pd.DataFrame(records)
csv_path = os.path.join(output_dir, "prompts.csv")
df.to_csv(csv_path, index=False)

print("✅ prompts.csv saved:", csv_path)

# =========================================================
# ZIP OUTPUT (Kaggle download)
# =========================================================
zip_path = "output.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive("output", "zip", output_dir)
print("✅ Images zipped → output.zip")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Using device: cuda


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

... (5 outputs left out)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

It seems like you have activated model offloading by calling `enable_model_cpu_offload`, but are now manually moving the pipeline to GPU. It is strongly recommended against doing so as memory gains from offloading are likely to be lost. Offloading automatically takes care of moving the individual components vae, text_encoder, text_encoder_2, tokenizer, tokenizer_2, unet, image_encoder, feature_extractor, scheduler to GPU when needed. To make sure offloading works as expected, you should consider moving the pipeline back to CPU: `pipeline.to('cpu')` or removing the move altogether if you use offloading.


✅ prompts.csv saved: output/prompts.csv
✅ Images zipped → output.zip


In [3]:
# Define the prompt and seed
# PROMPT = "a dog and a cat"
# SEED = 42

# Generate the image (we modify the function slightly to return the final image)
def generate_image_return(prompt, seed, idx,negative_prompt=None):
    generator = torch.Generator(device=device).manual_seed(seed)

    # Base pipeline → latent image
    latent = pipe(
        prompt=prompt,
        num_inference_steps=n_steps,
        denoising_end=high_noise_frac,
        guidance_scale=guidance_scale,
        generator=generator,
        output_type="latent",
        negative_prompt=negative_prompt
    ).images

    # Refiner pipeline → final image
    image = refiner(
        prompt=prompt,
        num_inference_steps=n_steps,
        denoising_start=high_noise_frac,
        guidance_scale=guidance_scale,
        generator=generator,
        image=latent
    ).images[0]

    # Save image
    filename = f"generated_{idx:05d}.png"
    image.save(os.path.join(output_dir, filename))

    # Save record including seed
    records.append({
        "image": filename,
        "prompt": prompt,
        "seed": seed
    })

    # Clean up
    del latent
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return image  # ✅ Return the PIL image

# # Generate and display
# image = generate_image_return(prompt=PROMPT, seed=SEED, idx=1)
# display(image)


In [5]:
import os
import torch
from pathlib import Path
from IPython.display import display
from diffusers import StableDiffusionXLImg2ImgPipeline

# ===============================
# ENTRY POINT
# ===============================
if __name__ == "__main__":

    # List of prompts to evaluate for compositional logic
    prompts = [
            "A ball roughly next to a basket"
    ]
    
    # Note: Ensure get_token_indices extracts the correct nouns for each prompt
    # Here we define the tracking words per prompt index
    tracking_words = [
        ["ball", "basket"]
    ]

    negative_prompt = (
        "same size object, equal size, merged objects, "
        "single subject, blurry, low quality, cartoon, illustration, "
        "3d render, painting, text, watermark"
    )

    # Generate seeds from 42 to 142 inclusive (101 total)
    seeds = [142,742,842]

    output_root = Path("./aesthetic_Batch")
    output_root.mkdir(exist_ok=True)

    for p_idx, prompt_text in enumerate(prompts):
        # Create a subfolder for each specific prompt to avoid clutter
        prompt_folder_name = prompt_text[:30].replace(" ", "_")
        prompt_path = output_root / f"prompt_{p_idx}_{prompt_folder_name}"
        prompt_path.mkdir(exist_ok=True)
        
        words = tracking_words[p_idx]

        print(f"\n🚀 Processing Prompt {p_idx+1}/{len(prompts)}: {prompt_text}")

        for seed_value in seeds:
            print(f"Generating Seed: {seed_value}...", end="\r")

            # ===============================
            # Stage 1 — FUZZY GENERATION
            # ===============================
            image = generate_image_return(prompt=prompt_text, seed=seed_value, negative_prompt=negative_prompt,idx=1)

            if image is None:
                continue

            # Save Stage-1 image
            s1_filename = f"seed_{seed_value}_stage1.png"
            image.save(prompt_path / s1_filename)


    print(f"\n✨ Batch complete. All images saved in: {output_root.absolute()}")


🚀 Processing Prompt 1/1: A ball roughly next to a basket


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


✨ Batch complete. All images saved in: /kaggle/working/aesthetic_Batch
